# L13 · DAPO and Stable Reasoning RL

## Goal

- isolate DAPO's four components
- compare Dr. GRPO reduction
- distinguish GSPO's sequence ratio

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L13:toy:42").hexdigest()
print(f"lesson=L13 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L13 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:1d3b156ef3789b22828f7a1491201aead31d8baa5d694383347d979efd7b9fa8 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: GRPO → **DAPO, Dr.GRPO, and GSPO** → stability evaluation

$$L_{DAPO}=L_{clip\text{-}higher}+L_{dynamic\ sampling}+L_{token\text{-}level}+L_{overlong}$$

DAPO is not one magic equation; it combines asymmetric clipping, dynamic sampling of informative groups, token-level loss, and overlong shaping. Dr.GRPO simplifies normalization, while GSPO aggregates token changes into a sequence-level ratio.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** Do groups with rewards `[0,0]` or `[1,1]` survive dynamic sampling? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>No. They contain no within-group comparison signal, so only `[0,1]` and `[1,0]` remain.</details>

In [2]:
from rl_study.algorithms.dapo import dynamic_sampling_filter, overlong_reward_shaping
from rl_study.algorithms.grpo import dr_grpo_advantages, gspo_sequence_loss
candidate_rewards = torch.tensor([[0., 0.], [0., 1.], [1., 1.], [1., 0.]])
dynamic = dynamic_sampling_filter(candidate_rewards, required_groups=2)
penalties = overlong_reward_shaping(
    torch.tensor([5, 6, 8, 10]), max_response_length=10, buffer_length=4
)
current = torch.log(torch.tensor([[2., 8.], [1., 1.]]))
mask = torch.ones_like(current, dtype=torch.bool)
gspo = gspo_sequence_loss(
    current, torch.zeros_like(current), torch.zeros_like(current),
    torch.tensor([1., -1.]), mask, clip_low=10., clip_high=10.
)
print({"dynamic_indices": dynamic.selected_group_indices.tolist(),
       "overlong_penalty": penalties.tolist(),
       "dr_adv": dr_grpo_advantages(candidate_rewards[1:2]).tolist(),
       "gspo_ratio": gspo.ratio.tolist()})

{'dynamic_indices': [1, 3], 'overlong_penalty': [-0.0, -0.0, -0.5, -1.0], 'dr_adv': [[-0.5, 0.5]], 'gspo_ratio': [4.0, 1.0]}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Separate functions make each stabilization component ablatable. Treating a paper recipe as one name hides reduction and masking differences.

**Common trap:** An off-by-one overlong buffer can jump to -1 just before max length. Analytic tests pin the start, midpoint, and end boundaries. Regression tests: `test_overlong_reward_shaping_boundaries`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert dynamic.selected_group_indices.tolist() == [1, 3]
assert penalties.tolist() == [0.0, 0.0, -0.5, -1.0]
print("checks=passed")

checks=passed


**Recall:** Which of DAPO's four components directly targets sample efficiency? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** Dynamic sampling retained groups 1 and 3, and overlong penalty moved 0→-0.5→-1. The GSPO ratio is a sequence aggregate, not a token ratio.
- Executable checks: `test_overlong_reward_shaping_boundaries`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L14 moves the same APIs toward public small models and GPU servers while auditing download, memory, and framework boundaries.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

[Implementation note](../../docs/algorithms/dapo.md) · [Course map](../../docs/course-map.en.md)

## Sources

- `dapo-2025` — `docs/sources.yml`
- `dr-grpo-2025` — `docs/sources.yml`
- `gspo-2025` — `docs/sources.yml`
- `repo-understand-r1-zero` — `docs/sources.yml`
- `framework-verl` — `docs/sources.yml`